# 第4章 第一个程序 + 性能分析

## 本章导读

在第 2、3 章中，我们用数组加法认识了线程的分工和数据的去向。现在把这个例子写成一个完整程序：给定两个等长数组，让 GPU 计算对应元素的和，再把结果取回来检查。

完成计算后，我们还想知道它用了多长时间。这个问题需要先回答两件事：我们测的是哪一个实现，计时从哪里开始、到哪里结束？本章先用 HIP 看清程序的组成，再用 Python 学习测量方法，最后解释测得的时间与有效带宽。

| 文件 | 本章用它做什么 |
| --- | --- |
| `code/part0-intro/chapter4/vector_add.hip` | 手写 HIP 向量加法，编译、运行并检查结果 |
| `code/part0-intro/chapter4/benchmark_vector_add.py` | 测量 **PyTorch 实现**的向量加法，学习预热、重复和 GPU event |

两个程序完成相同的数学运算，但 Python 脚本**没有调用前面的 HIP 文件**。它们分别帮助我们理解“怎样写”和“怎样测”，不能把后者的时间当成前者的性能。


本 Notebook 按单元执行配套程序，时间与设备名使用本次运行输出。教程参考数据来自 9070 XT；其他设备只能用自己的输出做平台内比较。

## 4.1 准备环境

先完成第 1 章的环境验证。下方单元检测当前 GPU 架构并定位仓库；随后从 `code/part0-intro/chapter4/` 编译和运行配套程序。

## 在云端运行本章

本教程以 RX 9070 XT（`gfx1201` / RDNA4）为讲解和参考环境。云端结果用于验证代码并观察同一平台内的变化，不应与参考数据或其他 GPU 的绝对性能直接比较。

云平台已预装 ROCm、PyTorch 和基础编译工具，可跳过本地的 `uv sync` 与环境激活步骤；本地读者仍按原步骤准备环境。本章所需的额外依赖会在章节内单独提示。

请根据当前 `rocminfo` 输出选择编译架构；未识别时先检查环境。

In [ ]:
# 检测当前 GPU 架构
import os
import re
import subprocess
import sys

ARCH_DETECT_TIMEOUT_S = 10
COMPILE_TIMEOUT_S = 120
SMOKE_TIMEOUT_S = 120
FULL_TIMEOUT_S = 300
SUPPORTED_ARCHES = {"gfx1100", "gfx1151", "gfx1201"}
GPU_AGENT_BLOCK_RE = re.compile(
    r"(?ms)^\s*Agent\s+\d+\s*$.*?(?=^\s*Agent\s+\d+\s*$|\Z)"
)
GPU_AGENT_NAME_RE = re.compile(r"(?m)^\s*Name:\s*(gfx[0-9a-z]+)\s*$")
WAVEFRONT_SIZE_RE = re.compile(r"(?m)^\s*Wavefront Size:\s*(\d+)\s*$")


def _gpu_agent_details(rocminfo_stdout):
    """Return gfx names and optional wavefront sizes from GPU Agent blocks only."""
    candidates = {}
    for block in GPU_AGENT_BLOCK_RE.findall(rocminfo_stdout):
        if not re.search(r"(?m)^\s*Device Type:\s*GPU\s*$", block):
            continue
        wavefront_match = WAVEFRONT_SIZE_RE.search(block)
        wavefront_size = int(wavefront_match.group(1)) if wavefront_match else None
        for name in GPU_AGENT_NAME_RE.findall(block):
            candidates[name] = wavefront_size
    return candidates


def detect_architecture():
    override = os.environ.get("HELLO_GPU_ARCH", "").strip()
    if override:
        if override not in SUPPORTED_ARCHES:
            raise RuntimeError(
                f"HELLO_GPU_ARCH 仅支持 {sorted(SUPPORTED_ARCHES)}；实际值={override!r}"
            )
        return override, "override", None

    try:
        result = subprocess.run(
            ["rocminfo"],
            capture_output=True,
            text=True,
            timeout=ARCH_DETECT_TIMEOUT_S,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"rocminfo 超时（timeout={ARCH_DETECT_TIMEOUT_S}s）\n"
            f"stdout:\n{exc.stdout or ''}\n"
            f"stderr:\n{exc.stderr or ''}"
        ) from exc
    except FileNotFoundError as exc:
        raise RuntimeError(
            f"rocminfo 无法启动（returncode=NOT_STARTED）\nstderr:\n{exc}"
        ) from exc

    if result.returncode != 0:
        raise RuntimeError(
            f"rocminfo 失败（returncode={result.returncode}）\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

    candidates = _gpu_agent_details(result.stdout)
    if len(candidates) != 1:
        raise RuntimeError(
            "rocminfo 必须恰好报告一个 GPU Agent Name: gfx...；"
            f"实际候选={sorted(candidates) or 'none'}"
        )
    arch, wavefront_size = next(iter(candidates.items()))
    return arch, "rocminfo", wavefront_size


def run_checked(command, *, cwd, label, timeout=COMPILE_TIMEOUT_S):
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            cwd=cwd,
            timeout=timeout,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"{label} 超时（timeout={timeout}s；returncode=TIMEOUT）\n"
            f"stdout:\n{exc.stdout or ''}\n"
            f"stderr:\n{exc.stderr or ''}"
        ) from exc
    except FileNotFoundError as exc:
        raise RuntimeError(
            f"{label} 无法启动（returncode=NOT_STARTED）\nstderr:\n{exc}"
        ) from exc
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} 失败（returncode={result.returncode}）\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )
    return result


arch, arch_source, wavefront_size = detect_architecture()
if arch not in SUPPORTED_ARCHES:
    raise RuntimeError(
        f"当前架构 {arch} 暂无本章对应 vector-add 说明；支持集合={sorted(SUPPORTED_ARCHES)}"
    )
print(f"arch={arch}; arch_source={arch_source}")
if arch_source == "override":
    print("wavefront_size=unknown（override 不证明硬件；以运行结果为准）")
elif wavefront_size is None:
    print("wavefront_size=unknown（GPU Agent 未报告；以运行结果为准）")
else:
    print(f"wavefront_size={wavefront_size}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

## 4.2 从向量加法到 HIP 程序

本节将同下标相加的规则，逐步放进一个可以编译运行的程序。

### 4.2.1 定义输入和输出

给定长度为 $N$ 的数组 $a$ 和 $b$，我们要得到同样长度的数组 $c$：

$$
c_i = a_i + b_i,\qquad i=0,1,\ldots,N-1.
$$

为了让结果容易检查，示例将 $a$ 的每个元素设为 1，$b$ 的每个元素设为 2。因此无论数组多长，$c$ 的每个元素都应该是 3。这里不需要矩阵运算或归约，只要把每个位置算对即可。

`vector_add.hip` 的 `main` 函数先准备主机端数组：

```cpp
  const int n = 1 << 20;
  const size_t bytes = n * sizeof(float);

  std::vector<float> h_a(n, 1.0f);
  std::vector<float> h_b(n, 2.0f);
  std::vector<float> h_c(n, 0.0f);
```

`1 << 20` 是 C++ 中的左移表达式，在这里等于 $2^{20}=1{,}048{,}576$。`n` 表示元素个数，`bytes` 表示**一个数组**需要多少字节。`std::vector<float>` 可以先理解为“保存浮点数的、长度可指定的数组”，`1.0f` 中的 `f` 表示 `float` 常量。

### 4.2.2 在主机和设备之间传递数据

主机（Host）指 CPU 一侧，设备（Device）指 GPU 一侧。本例使用两套存储：`h_a`、`h_b`、`h_c` 在主机内存中，`d_a`、`d_b`、`d_c` 指向 GPU 显存中的数组。变量名里的 `h_` 和 `d_` 是帮助阅读的命名约定。

先为 GPU 上的三个数组分配空间，再复制两个输入：

```cpp
  float* d_a = nullptr;
  float* d_b = nullptr;
  float* d_c = nullptr;

  HIP_CHECK(hipMalloc(&d_a, bytes));
  HIP_CHECK(hipMalloc(&d_b, bytes));
  HIP_CHECK(hipMalloc(&d_c, bytes));

  HIP_CHECK(hipMemcpy(d_a, h_a.data(), bytes, hipMemcpyHostToDevice));
  HIP_CHECK(hipMemcpy(d_b, h_b.data(), bytes, hipMemcpyHostToDevice));
```

`float*` 是一个指针，用来保存数组起始位置。`hipMalloc` 分配显存，并把得到的地址写入指针变量；`hipMemcpy` 的前三个参数依次是目标位置、来源位置和字节数。`h_a.data()` 给出主机数组的起始地址。

`HIP_CHECK` 是本文件定义的错误检查宏。先把它理解为一个检查步骤：HIP 调用失败时打印错误并退出，避免带着错误继续运行。完整定义保留在源码中。

如 数据流程图 所示，拷贝、GPU 计算和结果检查发生在不同位置。`hipMalloc` 只负责分配空间，数据复制由 `hipMemcpy` 完成。

![主机准备两个输入，复制到设备后执行加法，再把输出复制回主机检查](../../docs/part0-intro/chapter4/images/vector-add-data-path.svg)

*本例的数据流程：输入从主机传到设备，计算结果从设备传回主机；分配空间与复制数据是不同操作。*

### 4.2.3 每个线程完成一次加法

现在来看运行在 GPU 上的核函数：

```cpp
__global__ void vector_add(const float* a, const float* b, float* c, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx < n) {
    c[idx] = a[idx] + b[idx];
  }
}
```

`__global__` 表示这个函数由主机发起调用、在 GPU 上执行。`const float* a` 和 `const float* b` 表示核函数通过这些指针读取输入，不修改输入元素；`float* c` 指向可写的输出。

第 2 章推导过全局下标。这里把同一公式写成代码：`blockIdx.x` 是块编号，`blockDim.x` 是每块线程数，`threadIdx.x` 是块内线程编号。每个线程得到自己的 `idx`，因此同一段代码会访问不同位置。

例如每块有 256 个线程时，块 2 中的线程 5 处理的位置是 $2\times256+5=517$，它执行 `c[517] = a[517] + b[517]`。线程编号图 只展开需要观察的编号，其余线程省略。

![块 2 中的线程 5 通过 2 乘 256 加 5 得到全局下标 517](../../docs/part0-intro/chapter4/images/thread-index.svg)

*块编号与块内编号共同确定数组位置；启动的全部线程不要求同时驻留或同时执行。*

`if (idx < n)` 则限制有效下标。如果输入长度不能被每块线程数整除，最后一块仍会启动完整数量的线程；多出的线程不应访问数组。这个判断让同一个核函数也能处理这样的尾部。

### 4.2.4 启动、等待和检查

主机端选择每块 256 个线程，并计算需要的块数：

```cpp
  const int threads = 256;
  const int blocks = (n + threads - 1) / threads;
  vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
  HIP_CHECK(hipGetLastError());
  HIP_CHECK(hipDeviceSynchronize());

  HIP_CHECK(hipMemcpy(h_c.data(), d_c, bytes, hipMemcpyDeviceToHost));
```

整数除法会向下取整，所以分子先加 `threads - 1`，得到覆盖全部元素所需的块数。本例中 $1{,}048{,}576/256=4096$，刚好整除。`<<<blocks, threads>>>` 指定启动规模，后面的圆括号才是传给核函数的四个参数。

启动调用返回时，GPU 的计算可能还没有完成。`hipGetLastError()` 检查启动错误，`hipDeviceSynchronize()` 等待设备完成工作；之后将 `d_c` 拷回 `h_c`，主机才能用结果进行检查。程序最后用 `hipFree` 释放三块显存。


下面的单元编译并运行完整 `vector_add.hip`。`hipcc` 是编译器，`-O2` 开启优化，`-o` 指定输出路径。

In [ ]:
chapter4_dir = REPO_ROOT / "code/part0-intro/chapter4"
vector_add_hip = chapter4_dir / "vector_add.hip"
vector_add_bin = chapter4_dir / "vector_add"

compile_result = run_checked(
    ["hipcc", f"--offload-arch={arch}", "-O2",
     str(vector_add_hip), "-o", str(vector_add_bin)],
    cwd=chapter4_dir,
    label="vector_add.hip 编译",
    timeout=COMPILE_TIMEOUT_S,
)
print("编译成功")

# 先做一次小规模检查；run_checked 会在失败或超时时显示 returncode、stdout、stderr 并停止。
run_result = run_checked(
    [str(vector_add_bin)],
    cwd=chapter4_dir,
    label="Vector Add 小规模检查",
    timeout=SMOKE_TIMEOUT_S,
)
print("\n运行结果:")
print(run_result.stdout)

if "status: PASS" not in run_result.stdout:
    raise RuntimeError(
        "Vector Add 返回非 PASS，停止以避免继续消费旧结果；"
        f"stdout:\n{run_result.stdout}"
    )
print("\n✓ Vector Add 验证通过")

检查本次输出中的 `max_error` 与 `status`。当前固定输入要求每个结果为 3；编译成功和结果正确是不同的检查步骤。这里尚未测量手写 HIP kernel 的运行时间。

## 4.3 测量 PyTorch 向量加法

本节用较短的 Python 程序学习计时。我们继续计算向量加法，但改由 PyTorch 提供底层实现。使用 Python 可以先把注意力放在测量范围上；手写 HIP kernel 的详细计时与 profiling 将在下一篇展开。

### 4.3.1 固定需要测量的工作

PyTorch 的张量（tensor）是保存数值的多维数组，本章只用一维、`float32` 类型的张量。CPU 和 GPU 两条路径都预先创建输入和输出，再重复执行加法。以下是 GPU 路径的准备部分：

```python
    a = torch.ones(size, device="cuda", dtype=torch.float32)
    b = torch.full((size,), 2.0, device="cuda", dtype=torch.float32)
    c = torch.empty_like(a)
```

`ones` 将输入设为 1，`full` 将另一个输入设为 2，`empty_like` 只分配同形状的输出空间，其初始值未定义。后续每次计算都执行 `torch.add(a, b, out=c)`，把结果写到已经准备好的 `c` 中。

这里的 `device="cuda"` 并不表示使用了 NVIDIA 显卡。PyTorch 的 ROCm 版本沿用了 `torch.cuda` 接口和设备名，实际后端可通过 `torch.version.hip` 确认，详见 [PyTorch 的 HIP 说明](https://docs.pytorch.org/docs/2.11/notes/hip.html)。

我们将输入创建、输出分配和数值检查放在计时外，CPU 和 GPU 计时内都只安排加法。脚本默认将 PyTorch 的 CPU 算子线程数设为 1，并打印 CPU 型号与线程数。这样读到结果时，就知道 CPU 一栏采用了什么配置。

### 4.3.2 等待完成以后，读取时间

CPU 发出 GPU 操作后可以继续执行，因此“Python 已经执行到下一行”不等于“GPU 已经算完”。直接在提交前后读取 CPU 时钟，可能只量到提交过程的一部分。

GPU event 可以在同一条执行流中记录两个时间点：一个放在加法之前，一个放在加法之后。等结束 event 完成后，再读取它们的时间差：

```python
        start.record()
        torch.add(a, b, out=c)
        end.record()
        end.synchronize()
        times.append(start.elapsed_time(end))
```

其中 `start` 和 `end` 是启用了计时的 `torch.cuda.Event`。`record()` 将标记放入当前执行流，`end.synchronize()` 让主机等到结束标记完成，`elapsed_time` 返回两个标记之间经过的毫秒数。顺序见 event 时间线；同步之后做的是**读取时间差**，并非这时才开始计时。[PyTorch 的异步执行说明](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution)给出了同样的基本原则。

![GPU 依次经过开始 event、加法、结束 event，CPU 等待结束 event 后读取时间差](../../docs/part0-intro/chapter4/images/event-timing.svg)

*event 确定计时区间，同步保证区间已经结束。时间线表示执行顺序，不按实际耗时比例绘制。*

这个区间是设备侧的一段经过时间，不包含输入输出的主机与设备间拷贝，也不代表整个 Python 程序的耗时。对于很短的运算，提交节奏和设备状态也可能影响 event 间隔，不能仅凭一个 event 数字就断定核函数内部花了多少时间。

### 4.3.3 预热、重复和检查

正式测量前，脚本先执行 5 次加法作为预热（warmup），再重复测量 30 次。预热可以减轻首次执行和状态变化的影响，5 次是本例的配置，并不保证所有工作负载都已经稳定。

每轮得到一个时间，最后报告三种统计量：

| 输出字段 | 含义 | 本章怎样使用 |
| --- | --- | --- |
| `median_ms` | 将时间排序后的中间值 | 用作正文主要比较值 |
| `mean_ms` | 所有时间的平均值 | 与中位数一起观察波动 |
| `min_ms` | 这一组中最短的一次 | 作为补充，不视为保证无干扰的“真值” |

测量结束后，脚本检查输出的每个元素是否有限且等于 3；验证失败就报错，不打印通过状态。检查放在计时外，避免把求和或比较混入加法的时间。


下面运行 PyTorch 加法 benchmark，并解析本次实际输出。默认规模为 16,777,216 个 float32，warmup 5、repeat 30、CPU 算子线程数 1。

In [ ]:
benchmark_script = chapter4_dir / "benchmark_vector_add.py"

run_result = run_checked(
    [sys.executable, str(benchmark_script),
     "--size", "16777216",  # 2^24
     "--warmup", "5",
     "--repeat", "30",
     "--cpu-threads", "1"],
    cwd=chapter4_dir,
    label="Vector Add 完整 benchmark",
    timeout=FULL_TIMEOUT_S,
)
print("Benchmark 结果:")
print(run_result.stdout)


def parse_benchmark_output(stdout):
    """Parse the stable key:value lines emitted by benchmark_vector_add.py."""
    integer_fields = {"vector_size", "warmup", "repeat", "cpu_threads"}
    float_fields = {
        "cpu_mean_ms", "cpu_median_ms", "cpu_min_ms",
        "cpu_bandwidth_gb_s_by_min", "gpu_mean_ms", "gpu_median_ms",
        "gpu_min_ms", "gpu_bandwidth_gb_s_by_min",
        "cpu_bandwidth_gb_s_by_median", "gpu_bandwidth_gb_s_by_median",
    }
    parsed = {}
    for line in stdout.splitlines():
        key, separator, value = line.partition(":")
        key = key.strip()
        if not separator:
            continue
        value = value.strip()
        if key in integer_fields:
            parsed[key] = int(value)
        elif key in float_fields:
            parsed[key] = float(value)
        elif key in {"torch", "hip", "cuda_available", "device_name", "cpu_name",
                    "cpu_timing", "gpu_timing", "cpu_validation", "gpu_validation", "status"}:
            parsed[key] = value

    required = {
        "device_name", "vector_size", "warmup", "repeat", "cpu_threads",
        "cpu_name", "hip", "cpu_timing", "gpu_timing", "cpu_validation", "gpu_validation",
        "cpu_mean_ms", "cpu_median_ms", "cpu_min_ms",
        "cpu_bandwidth_gb_s_by_min", "gpu_mean_ms", "gpu_median_ms",
        "gpu_min_ms", "gpu_bandwidth_gb_s_by_min",
        "cpu_bandwidth_gb_s_by_median", "gpu_bandwidth_gb_s_by_median", "status",
    }
    missing = sorted(required - parsed.keys())
    if missing:
        raise RuntimeError(f"benchmark 输出缺少字段: {missing}")
    if parsed["status"] != "PASS":
        raise RuntimeError(f"benchmark 未通过: status={parsed['status']}")
    for field in ("cpu_validation", "gpu_validation"):
        if not parsed[field].startswith("PASS ("):
            raise RuntimeError(f"结果验证未通过: {field}={parsed[field]}")
    return parsed


benchmark_result = parse_benchmark_output(run_result.stdout)
print("\n已解析当前平台结果；4.4 节将直接使用本次输出，不使用固定平台数值。")
print("\n关键指标说明:")
print("- warmup: 正式计时前的预热次数；不保证任何配置都已稳定")
print("- repeat: 重复测量次数，用于统计分析")
print("- median_ms: 本章主要统计量；mean/min 用于补充观察波动")
print("- bandwidth_gb_s_by_median: 算法有效字节数 12N / 中位数时间")

## 4.4 解释测量结果

### 4.4.1 先读时间和计时范围

下面直接读取刚才解析的结果，同时打印 CPU 型号、线程数以及两个计时范围。CPU 与 GPU 都执行预分配输出的加法，但前者是 CPU 时钟包围的调用，后者是 GPU event 间隔；不能把二者的比值当作完整应用的加速比。

In [ ]:
def print_current_platform_result(result):
    print("当前平台 PyTorch 加法实测")
    for key in ("device_name", "cpu_name", "cpu_threads", "hip", "vector_size", "warmup", "repeat"):
        print(f"- {key}: {result[key]}")
    print(f"- detected_arch: {arch}")
    print(f"- CPU 计时: {result['cpu_timing']}")
    print(f"- GPU 计时: {result['gpu_timing']}")
    for statistic in ("median", "mean", "min"):
        print(f"- {statistic}: CPU={result[f'cpu_{statistic}_ms']:.6f} ms, "
              f"GPU={result[f'gpu_{statistic}_ms']:.6f} ms")
    print(f"- CPU 输出验证: {result['cpu_validation']}")
    print(f"- GPU 输出验证: {result['gpu_validation']}")


print_current_platform_result(benchmark_result)

### 4.4.2 从算法数据量到有效带宽

每个 float32 输出对应两次读取和一次写入：读 a[i] 的 4 字节、读 b[i] 的 4 字节、写 c[i] 的 4 字节。因此有效数据量为 12N 字节。这个模型没有测量缓存或显存中的物理流量。

下面使用本次实际元素个数计算字节数。GB 按 10⁹ 字节计算，MiB 按 2²⁰ 字节计算。

In [ ]:
N = benchmark_result["vector_size"]
bytes_per_element = 4  # float32
reads = 2 * N * bytes_per_element
writes = N * bytes_per_element
total_bytes = reads + writes
flops = N  # 每个元素一次加法
arithmetic_intensity = flops / total_bytes

print(f"每个数组: {N * bytes_per_element / 2**20:g} MiB")
print(f"算法读取: {reads:,} Byte")
print(f"算法写入: {writes:,} Byte")
print(f"算法有效数据量: {total_bytes:,} Byte")

用算法有效数据量除以 GPU 中位数时间，可得到本次有效带宽：

$$BW_{\mathrm{effective}} = \frac{12N}{t}.$$

`gpu_median_ms` 的单位是毫秒，先除以 1000 换成秒；再将 Byte/s 除以 10⁹，得到 GB/s。下面同时打印重算值与脚本输出，检查单位是否一致。

In [ ]:
actual_gpu_median_ms = benchmark_result["gpu_median_ms"]
actual_bandwidth_gb_s = total_bytes / (actual_gpu_median_ms / 1000) / 1e9
reported_bandwidth_gb_s = benchmark_result["gpu_bandwidth_gb_s_by_median"]

print(f"GPU 中位数: {actual_gpu_median_ms:.6f} ms")
print(f"重算有效带宽: {actual_bandwidth_gb_s:.3f} GB/s")
print(f"脚本有效带宽: {reported_bandwidth_gb_s:.3f} GB/s")
print("时间字段经过舍入，重算值的末位可能与原始统计略有差异。")
print("这不是硬件计数器测得的物理显存带宽，也不是整个程序的吞吐。")

### 4.4.3 一次加法为什么要关心数据移动

每个元素只做一次加法，却对应 12 字节的算法有效数据。于是算术强度为 1/12，约 0.0833 FLOP/Byte。对足够大的向量，通常先关注数据访问；对短运算，启动、提交节奏和缓存也可能显著影响时间。

即使把有效带宽与当前 GPU 的官方显存带宽规格相除，得到的也只是模型下的比值。它不能当作显存控制器利用率，更不能据此算出剩余优化空间。当前 Notebook 不自动猜测设备的峰值规格。

## 4.5 从算子时间到程序时间

本次 GPU event 前已经创建了输入和输出空间。如果真实任务的输入位于 CPU 内存，还需要计算输入拷贝、提交、等待和结果拷回的总时间。是否使用 GPU 应结合规模、数据位置和多步计算中的复用情况，通过完整流程的测量判断；没有通用的元素数分界。

## 4.6 记录与练习

先保留本次实际参数和输出，再改变一个条件重跑。预热与重复次数是实验配置，不能保证一组时间一定稳定。下面先生成一份简短的算法模型说明，再整理运行记录。

In [ ]:
print(f"算法算术强度: {arithmetic_intensity:.4f} FLOP/Byte")
print("阅读顺序：先看 median，再结合 mean/min 观察波动。")
print("预热在正式采样前；同步返回后读取已经记录好的 event 时间差。")
print("结果验证在计时外；PASS 必须来自对实际输出的检查。")

实验记录至少应包含日期、设备、系统与软件版本、源码版本、输入规模、线程配置、预热与重复次数、计时范围和正确性结果。下方表格来自本次运行；日期、操作系统和源码版本需要另外补到个人记录中。

In [ ]:
print("## 本次实测记录")
print("| 项目 | 数值 |")
print("| ---- | ---- |")
for key in ("device_name", "cpu_name", "cpu_threads", "hip", "vector_size", "warmup", "repeat"):
    print(f"| {key} | {benchmark_result[key]} |")
print(f"| CPU 中位数 | {benchmark_result['cpu_median_ms']:.6f} ms |")
print(f"| GPU 中位数 | {benchmark_result['gpu_median_ms']:.6f} ms |")
print(f"| GPU 算法有效带宽（median） | {benchmark_result['gpu_bandwidth_gb_s_by_median']:.3f} GB/s |")
print(f"| CPU 范围 | {benchmark_result['cpu_timing']} |")
print(f"| GPU 范围 | {benchmark_result['gpu_timing']} |")
print(f"| 输出检查 | {benchmark_result['status']} |")

### 练习

1. 每块 256 个线程、输入长度 1000 时，手算需要几个 block，以及最后一块的有效线程编号。
2. 说明为什么结束 event 已经提交，并不保证可以立即读取它的时间戳。
3. 将 benchmark 运行单元中的 `--size` 分别改为 `1048576`、`67108864`，重新执行后续统计单元。比较 mean、median 和 min，不预设规模越大就一定更有效率。
4. 用本次时间手算 12N/t，并说明结果为什么不是物理显存流量。

## 本章小结

- 从主机输入、设备空间、拷贝、核函数启动、等待和检查，组成完整 HIP 示例。
- 手写 HIP 程序用于验证结果；PyTorch benchmark 用于学习计时，两者是不同实现。
- CPU/GPU 计时内均执行预分配输出的加法，结果检查在计时外；仍要说明两个时钟各自覆盖的范围。
- GPU event 定义区间，同步保证区间结束；用 median 配合 mean/min 观察一组测量。
- 有效带宽来自算法数据量和实测时间，不能直接视为显存控制器利用率或应用加速比。

## 延伸阅读

- [PyTorch：HIP（ROCm）接口说明](https://docs.pytorch.org/docs/2.11/notes/hip.html)
- [PyTorch：异步执行与计时](https://docs.pytorch.org/docs/stable/notes/cuda.html#asynchronous-execution)
- [《动手学深度学习》：异步计算](https://zh.d2l.ai/chapter_computational-performance/async-computation.html)